# PS2: Right-Party Contact Prediction (Perfect Pipeline)

This pipeline predicts Right-Party Contact by utilizing time-decay network responses and aggregate call behaviors derived from our deep EDA.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import os

print("Imports successful. Environment is ready.")

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"
print(f"Targeting data path: {base_path}")

In [ ]:
phones = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "phones.csv"))
dial_attempts = pd.read_csv(os.path.join(base_path, "shared", "dial_attempts.csv"))
verified = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "verified_contact_points.csv"))

print(f"Phones dataset: {phones.shape}")
print(f"Dial attempts dataset: {dial_attempts.shape}")
print(f"Verified targets dataset: {verified.shape}")

print("\nPeeking at network response counts:")
print(dial_attempts['network_response'].value_counts())

## 3. Feature Engineering with Time-Decay

In [ ]:
print("Converting verified_status to binary is_rpc target...")
verified['is_rpc'] = (verified['verified_status'] == 'borrower_number').astype(int)
print(f"Target class balance:\n{verified['is_rpc'].value_counts(normalize=True)}")

print("Formatting timestamps...")
dial_attempts['attempt_ts'] = pd.to_datetime(dial_attempts['attempt_ts'])

In [ ]:
print("Sorting dialing attempts chronologically to calculate recency metrics...")
dial_attempts = dial_attempts.sort_values('attempt_ts')
anchor_date = pd.to_datetime('2026-08-01')

In [ ]:
print("Aggregating dialing patterns by phone_id...")
# Aggregate with ratios as dictated by EDA
attempts_agg = dial_attempts.groupby('phone_id').agg(
    total_attempts=('attempt_id', 'count'),
    switched_off_count=('network_response', lambda x: (x == 'switched_off').sum()),
    answered_count=('talk_duration_s', lambda x: (x > 0).sum()),
    last_attempt_date=('attempt_ts', 'max'),
    avg_talk_time=('talk_duration_s', 'mean')
).reset_index()

print("Aggregation complete. Sample output:")
print(attempts_agg.head())

In [ ]:
print("Calculating network response ratios and time since last interaction...")
attempts_agg['switched_off_ratio'] = attempts_agg['switched_off_count'] / attempts_agg['total_attempts']
attempts_agg['answered_ratio'] = attempts_agg['answered_count'] / attempts_agg['total_attempts']
attempts_agg['days_since_last_attempt'] = (anchor_date - attempts_agg['last_attempt_date']).dt.days

print("Engineered features stats:")
print(attempts_agg[['switched_off_ratio', 'answered_ratio', 'days_since_last_attempt']].describe())

In [ ]:
print("Joining features with target dataframe...")
data = pd.merge(verified, phones, on='phone_id', how='left')
data = pd.merge(data, attempts_agg, on='phone_id', how='left')

print("Filling nulls for uncontacted phones...")
data = data.fillna({
    'total_attempts': 0, 
    'switched_off_ratio': 0, 
    'answered_ratio': 0, 
    'days_since_last_attempt': -1, 
    'avg_talk_time': 0
})

print(f"Final data shape before modeling: {data.shape}")

In [ ]:
features = ['total_attempts', 'switched_off_ratio', 'answered_ratio', 'days_since_last_attempt', 'avg_talk_time', 'priority_slot']
print("Using features:", features)
X = data[features]
y = data['is_rpc']

## 4. Modeling

In [ ]:
print("Performing 80-20 train/test split...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
print("Fitting HistGradientBoostingClassifier (optimised for robust tabular classification)...")
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=5, random_state=42)
clf.fit(X_train, y_train)
print("Model fit complete!")

In [ ]:
print("Predicting probabilities for ROC AUC...")
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

In [ ]:
print("\n--- Final Metrics ---")
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
auc = roc_auc_score(y_test, y_prob)
print(f"ROC AUC Score: {auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))